# Assignment 8 — Mass-Spring Rope

> **GAMES101 — Intro to Computer Graphics** (Lingqi Yan, UCSB).
> Course site: <https://sites.cs.ucsb.edu/~lingqi/teaching/games101.html>
>
> The course ships C++ starter code with Eigen + OpenCV. I'm doing the same tasks in
> Python notebooks so I can iterate on the math cell-by-cell. Notes at the top of each
> notebook are what I actually needed to remember to get the assignment out.

## Topic

The animation-block finale. A **rope** is a chain of point masses connected by
springs. Each frame we sum the spring forces + gravity, then step the state forward.
The whole assignment is about *which integrator you use to step time*:

- **Explicit (forward) Euler** — `v += a*dt; x += v*dt`. Simplest possible; also
  famously unstable for stiff springs. It gains energy on every step, so an
  undamped rope with `k=100, dt=1/60` explodes within about half a second.
  *This blow-up is the lesson. Screenshot it.*
- **Semi-implicit (symplectic) Euler** — use the *just-updated* velocity for the
  position update. One-line change, and it stops the blow-up.
- **Verlet** — position-based: `x_new = x + (x - x_last) + a*dt²`. Doesn't store
  velocity at all, damps very naturally, and plays nicely with hard constraints.

See: [Verlet integration](https://en.wikipedia.org/wiki/Verlet_integration), [Semi-implicit Euler](https://en.wikipedia.org/wiki/Semi-implicit_Euler_method), [Numerical stability of ODE integrators](https://en.wikipedia.org/wiki/Numerical_methods_for_ordinary_differential_equations).


In [1]:
import numpy as np

class Mass:
    def __init__(self, pos, pinned=False):
        self.pos = np.array(pos, dtype=float)
        self.last = self.pos.copy()
        self.vel = np.zeros(2)
        self.force = np.zeros(2)
        self.pinned = pinned

class Spring:
    def __init__(self, a, b, k):
        self.a, self.b, self.k = a, b, k
        self.rest = np.linalg.norm(a.pos - b.pos)

class Rope:
    def __init__(self, start, end, N, k, pinned=(0,)):
        self.masses = []
        for i in range(N):
            t = i / (N - 1)
            self.masses.append(Mass(start*(1-t) + end*t, pinned=(i in pinned)))
        self.springs = [Spring(self.masses[i], self.masses[i+1], k) for i in range(N-1)]

    def simulate_euler(self, dt, g):
        for s in self.springs:
            d = s.b.pos - s.a.pos
            L = np.linalg.norm(d)
            f = s.k * (d / L) * (L - s.rest)
            s.a.force += f
            s.b.force -= f
        for m in self.masses:
            if m.pinned:
                m.force[:] = 0
                continue
            m.force += g  # gravity per unit mass, treating m=1
            # explicit euler -- known unstable without damping
            m.vel += m.force * dt
            m.pos += m.vel * dt
            m.force[:] = 0


Yep, it explodes within ~30 frames at k=100, dt=1/60. That's the lesson.